## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [2]:
# CHANGED: Model to Groq model
MODEL = "openai/gpt-oss-120b"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [3]:
# UNCHANGED - still using HuggingFace embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [4]:
retriever = vectorstore.as_retriever()
llm = ChatGroq(temperature=0, model_name=MODEL)

### These LangChain objects implement the method `invoke()`

In [5]:
retriever.invoke("Who is Avery?")

[Document(id='31b88c75-0f18-4ec5-8ee1-27857142df3c', metadata={'doc_type': 'employees', 'source': 'knowledge-base/employees/Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and ada

In [6]:
llm.invoke("Who is Avery?")

AIMessage(content='I’m not sure which “Avery” you’re referring to—there are many people, characters, and entities with that name. Could you give me a bit more context? For example:\n\n- Are you thinking of a public figure (e.g., a musician, athlete, author, etc.)?\n- A fictional character from a book, TV show, movie, or video game?\n- Someone you know personally or a family member?\n- A brand, company, or product named Avery (like the label and office‑supply brand)?\n\nLet me know, and I’ll be happy to provide the information you’re looking for!', additional_kwargs={'reasoning_content': 'The user asks "Who is Avery?" No context. Could be a public figure, a name. Need to respond with clarification: ask for more context. According to policy, we can ask for clarification. Provide possible interpretations.'}, response_metadata={'token_usage': {'completion_tokens': 182, 'prompt_tokens': 75, 'total_tokens': 257, 'completion_time': 0.473678974, 'completion_tokens_details': {'reasoning_tokens'

## Time to put this together!

In [9]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [10]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [11]:
answer_question("Who is Averi Lancaster?", [])

'**Avery\u202fLancaster** (sometimes miss‑spelled as “Averi”) is one of the founding pillars of Insurellm.\n\n| Detail | Information |\n|--------|--------------|\n| **Full Name** | Avery\u202fLancaster |\n| **Date of Birth** | March\u202f15\u202f,\u202f1985 |\n| **Current Role** | Co‑Founder & Chief Executive Officer (CEO) |\n| **Location** | San\u202fFrancisco, California |\n| **Current Salary** | $225,000 per year |\n| **Company Tenure** | 2015\u202f–\u202fPresent (co‑founded Insurellm) |\n| **Career Highlights** | • Co‑founded Insurellm in 2015 and has steered the company to become a leading Insurance‑Tech provider.<br>• Recognized for innovative leadership and deep expertise in risk management, helping the firm break into the mainstream insurance market.<br>• Prior to Insurellm, served as Senior Product Manager (2013‑2015) at Innovate Insurance Solutions, where she launched groundbreaking insurance products for the tech sector. |\n\nIn short, Avery Lancaster is the visionary CEO wh

## What could possibly come next? 😂

In [12]:
gr.ChatInterface(answer_question).launch()

/home/jay/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!